In [1]:
# ==============================================================================
# Exploratory Data Analysis (EDA) for MNIST, Fashion-MNIST, and CIFAR-10
# Saves high-resolution .png plots into ./eda_outputs/ for GitHub & reports
# ==============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist, fashion_mnist, cifar10

# Create output directory
OUTPUT_DIR = "eda_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Class label mappings
LABELS = {
    "MNIST": [str(i) for i in range(10)],
    "Fashion-MNIST": [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ],
    "CIFAR-10": [
        "Airplane", "Automobile", "Bird", "Cat", "Deer",
        "Dog", "Frog", "Horse", "Ship", "Truck"
    ],
}

# Load datasets
(x_tr_m, y_tr_m), (x_te_m, y_te_m) = mnist.load_data()
(x_tr_f, y_tr_f), (x_te_f, y_te_f) = fashion_mnist.load_data()
(x_tr_c, y_tr_c), (x_te_c, y_te_c) = cifar10.load_data()

datasets = {
    "MNIST": (x_tr_m, y_tr_m.squeeze(), x_te_m, y_te_m.squeeze()),
    "Fashion-MNIST": (x_tr_f, y_tr_f.squeeze(), x_te_f, y_te_f.squeeze()),
    "CIFAR-10": (x_tr_c, y_tr_c.squeeze(), x_te_c, y_te_c.squeeze()),
}

# Print dataset summary statistics
print("=" * 75)
print(f"{'Dataset':<15} | {'Train Shape':<18} | {'Test Shape':<16} | {'Dtype':<8}")
print("=" * 75)
for name, (x_tr, _, x_te, _) in datasets.items():
    print(f"{name:<15} | {str(x_tr.shape):<18} | {str(x_te.shape):<16} | {str(x_tr.dtype):<8}")
print("=" * 75)


def save_sample_grid(name, x_data, y_data, filename, n_per_class=2):
    """Save a grid showing sample images across all 10 classes."""
    class_names = LABELS[name]
    fig, axes = plt.subplots(
        nrows=10, ncols=n_per_class,
        figsize=(n_per_class * 2.2, 16),
        squeeze=False
    )
    fig.suptitle(f"{name} — Sample Images per Class", fontsize=14, fontweight="bold", y=0.995)

    for cls_idx in range(10):
        indices = np.where(y_data == cls_idx)[0][:n_per_class]
        for col_idx in range(n_per_class):
            ax = axes[cls_idx, col_idx]
            img = x_data[indices[col_idx]]
            if img.ndim == 2:
                ax.imshow(img, cmap="gray", vmin=0, vmax=255)
            else:
                ax.imshow(img)
            ax.set_xticks([])
            ax.set_yticks([])
            if col_idx == 0:
                ax.set_ylabel(f"{cls_idx}: {class_names[cls_idx]}", rotation=0,
                              labelpad=75, va="center", fontsize=9, fontweight="bold")

    plt.tight_layout(rect=[0.08, 0, 1, 0.98])
    out_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {out_path}")


def save_class_distribution(name, y_train, y_test, filename):
    """Save a side-by-side bar chart of train vs. test class counts."""
    class_names = LABELS[name]
    x_pos = np.arange(10)
    width = 0.38

    train_counts = [np.sum(y_train == i) for i in range(10)]
    test_counts = [np.sum(y_test == i) for i in range(10)]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x_pos - width / 2, train_counts, width, label="Train Set")
    ax.bar(x_pos + width / 2, test_counts, width, label="Test Set")

    ax.set_title(f"{name} — Class Distribution (Train vs. Test)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Samples")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(class_names, rotation=30, ha="right")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {out_path}")


def save_pixel_intensity_hist(datasets_dict, filename):
    """Save pixel intensity distributions (0-255) for each dataset."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    fig.suptitle("Pixel Intensity Distributions Across Datasets", fontsize=14, fontweight="bold")

    for ax, (name, (x_tr, _, _, _)) in zip(axes, datasets_dict.items()):
        # Subsample up to 200,000 pixels for fast plotting
        flat_pixels = x_tr.ravel()
        if flat_pixels.size > 200_000:
            rng = np.random.default_rng(42)
            flat_pixels = rng.choice(flat_pixels, size=200_000, replace=False)

        ax.hist(flat_pixels, bins=32, range=(0, 255), alpha=0.85, edgecolor="black", linewidth=0.4)
        ax.set_title(f"{name} (mean={x_tr.mean():.1f}, std={x_tr.std():.1f})")
        ax.set_xlabel("Pixel Value (0–255)")
        ax.set_ylabel("Frequency")
        ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    out_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {out_path}")


# Generate all PNG visualizations
for ds_name, (x_tr, y_tr, x_te, y_te) in datasets.items():
    slug = ds_name.lower().replace("-", "_")
    save_sample_grid(ds_name, x_tr, y_tr, f"{slug}_samples.png", n_per_class=2)
    save_class_distribution(ds_name, y_tr, y_te, f"{slug}_class_dist.png")

save_pixel_intensity_hist(datasets, "pixel_intensity_comparison.png")

print("\nAll .png files saved successfully in:", os.path.abspath(OUTPUT_DIR))
print("Files generated:", sorted(os.listdir(OUTPUT_DIR)))

# Optional: Zip the output folder for easy download from Colab
# !zip -r eda_outputs.zip eda_outputs

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 271s 2us/step
Dataset         | Train Shape        | Test Shape       | Dtype   
MNIST           | (60000, 28, 28)    | (10000, 28, 28)  | uint8   
Fashion-MNIST   | (60000, 28, 28)    | (10000, 28, 28)  | uint8   
CIFAR-10        | (50000, 32, 32, 3) | (10000, 32, 32, 3) | uint8   
Saved: eda_outputs/mnist_samples.png
Saved: eda_outputs/mnist_class_dist.png
Saved: eda_outputs/fashion_mnist_samples.png
Saved: eda_outputs/fashion_mnist_class_dist.png
Saved: eda_outputs/cifar_10_samples.png
Saved: eda_outputs/cifar_10_class_dist.png
Saved: eda_outputs/pixel_intensity_comparison.png

All .png files saved successfully in: /content/eda_outputs
Files generated: ['cifar_10_class_dist.png', 'cifar_10_samples.png', 'fas